# Debug Label Masking Issue

Notebook ini membantu debug kegagalan training dimana loss=0.0000 dan eval_loss=NaN.

**Hipotesis**: Semua labels di-mask dengan -100, menyebabkan zero loss.

**Referensi**: [DataCollatorForSeq2Seq Documentation](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorForSeq2Seq)

In [1]:
import sys
sys.path.append('/content')

from src.finetuned.utils.model_loader import load_model_with_lora
from src.finetuned.data.dataset_loader import DatasetLoader
from transformers import DataCollatorForSeq2Seq
import torch

## Step 1: Load Model dan Tokenizer

In [2]:
peft_model, tokenizer = load_model_with_lora(
    model_name='Wikidepia/IndoT5-base',
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

print(f"Tokenizer pad_token_id: {tokenizer.pad_token_id}")
print(f"Tokenizer eos_token_id: {tokenizer.eos_token_id}")
print(f"Model device: {peft_model.device}")

Loading base model: Wikidepia/IndoT5-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


spiece.model:   0%|          | 0.00/777k [00:00<?, ?B/s]

✓ Base model loaded
✓ LoRA applied: r=8, alpha=16, target=['q', 'v']
  Trainable: 884,736 (0.30%)
  Total:     297,811,200
✓ Model device: cuda:0
  GPU allocated: 1.19 GB
Tokenizer pad_token_id: 0
Tokenizer eos_token_id: 1
Model device: cuda:0


## Step 2: Load Sample Data

In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted


In [12]:
# Setup paths dan copy dataset dari Drive
import os
import shutil

DRIVE_ROOT = '/content/drive/MyDrive/dataset_aqg'
TASK_DIR = '/content/dataset_aqg/dataset-task-spesifc/'

# Copy dataset dari Drive jika belum ada
if not os.path.exists(TASK_DIR + 'train.jsonl'):
    drive_task = f'{DRIVE_ROOT}/dataset-task-spesifc'
    os.makedirs(TASK_DIR, exist_ok=True)
    for f in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy(f'{drive_task}/{f}', f'{TASK_DIR}{f}')
    print('✓ Dataset copied from Drive')
else:
    print('✓ Dataset already exists')

# Load dataset
loader = DatasetLoader()
train_dataset = loader.load_dataset(TASK_DIR, split='train')
print(f"\nLoaded {len(train_dataset)} samples")

# Ambil sample pertama
sample = train_dataset[0]
print(f"\nSample input length: {len(sample['input'])} chars")
print(f"Sample target length: {len(sample['target'])} chars")
print(f"\nInput preview : {sample['input']}...")
print(f"\nTarget preview : {sample['target']}...")

✓ Dataset already exists
✓ Loaded 876 entries from /content/dataset_aqg/dataset-task-spesifc/train.jsonl

Loaded 876 samples

Sample input length: 973 chars
Sample target length: 422 chars

Input preview : Konteks: ### Perbandingan Penggunaan Memori

```python
import numpy
import sys

var_list = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
var_array = numpy.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("Ukuran keseluruhan elemen list dalam bytes =", sys.getsizeof(var_list) * len(var_list))
print("Ukuran keseluruhan elemen NumPy dalam bytes =", var_array.size * var_array.itemsize)

"""
Output:
Ukuran keseluruhan elemen list dalam bytes = 240
Ukuran keseluruhan elemen NumPy dalam bytes = 72
"""
```
Dengan matriks yang sama, NumPy hanya menggunakan **72 bytes** dibanding list Python yang menggunakan **240 bytes** — inilah alasan banyak programmer memilih NumPy untuk memproses matriks. > **Catatan:** Seluruh materi pada modul ini akan menggunakan list Python untuk mengimplementasikan matriks, agar 

## Step 3: Test Tokenization (TANPA DataCollator)

Verifikasi bahwa tokenization menghasilkan labels yang valid.

In [14]:
# Tokenize input
input_tokens = tokenizer(
    sample['input'],
    max_length=512,
    truncation=True
)

# Tokenize target dengan text_target (PENTING untuk T5!)
label_tokens = tokenizer(
    text_target=sample['target'],
    max_length=512,
    truncation=True
)

print("=== HASIL TOKENIZATION ===")
print(f"\nInput IDs length: {len(input_tokens['input_ids'])}")
print(f"Input IDs sample (20 pertama): {input_tokens['input_ids'][:20]}")
print(f"\nLabel IDs length: {len(label_tokens['input_ids'])}")
print(f"Label IDs sample (20 pertama): {label_tokens['input_ids'][:20]}")

# Cek padding (seharusnya TIDAK ada karena belum di-pad)
input_pad_count = sum(1 for x in input_tokens['input_ids'] if x == tokenizer.pad_token_id)
label_pad_count = sum(1 for x in label_tokens['input_ids'] if x == tokenizer.pad_token_id)

print(f"\nInput padding tokens: {input_pad_count} (seharusnya 0)")
print(f"Label padding tokens: {label_pad_count} (seharusnya 0)")

# Cek non-zero labels
non_zero_labels = sum(1 for x in label_tokens['input_ids'] if x != 0)
print(f"\nNon-zero label tokens: {non_zero_labels} / {len(label_tokens['input_ids'])}")

if non_zero_labels == 0:
    print("\n❌ MASALAH: Semua labels adalah 0 (padding)!")
    print("   Ini berarti tokenization gagal.")
else:
    print("\n✓ Labels mengandung token non-zero (BAGUS)")

=== HASIL TOKENIZATION ===

Input IDs length: 319
Input IDs sample (20 pertama): [2777, 5561, 39, 2892, 18209, 18209, 926, 16135, 30, 12489, 24532, 11, 10347, 10347, 10347, 258, 22502, 7676, 120, 11]

Label IDs length: 201
Label IDs sample (20 pertama): [926, 369, 23, 30, 39, 15506, 1489, 10059, 10, 138, 19133, 211, 22502, 21, 12942, 8, 2046, 43, 1620, 614]

Input padding tokens: 0 (seharusnya 0)
Label padding tokens: 0 (seharusnya 0)

Non-zero label tokens: 201 / 201

✓ Labels mengandung token non-zero (BAGUS)


## Step 4: Test DataCollator Behavior

Verifikasi bahwa DataCollator HANYA mem-mask padding tokens, BUKAN semua tokens.

**Sesuai dokumentasi HuggingFace**:
- `label_pad_token_id=-100`: Hanya padding tokens yang di-mask dengan -100
- `padding=True`: Dynamic padding ke panjang maksimum dalam batch
- `pad_to_multiple_of=8`: Pad ke kelipatan 8 untuk efisiensi GPU

In [15]:
# Buat DataCollator sesuai dokumentasi
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=peft_model,
    label_pad_token_id=-100,  # Mask padding dengan -100
    padding=True,  # Dynamic padding
    pad_to_multiple_of=8  # Untuk efisiensi GPU
    # TIDAK menggunakan max_length - biarkan dynamic padding bekerja
)

# Buat batch dengan 2 samples (panjang berbeda)
batch_samples = [
    {
        'input_ids': input_tokens['input_ids'],
        'attention_mask': input_tokens['attention_mask'],
        'labels': label_tokens['input_ids']
    },
    {
        'input_ids': input_tokens['input_ids'][:50],  # Sample lebih pendek
        'attention_mask': input_tokens['attention_mask'][:50],
        'labels': label_tokens['input_ids'][:30]
    }
]

print("=== SEBELUM DATACOLLATOR ===")
print(f"Sample 1 - Input length: {len(batch_samples[0]['input_ids'])}")
print(f"Sample 1 - Label length: {len(batch_samples[0]['labels'])}")
print(f"Sample 2 - Input length: {len(batch_samples[1]['input_ids'])}")
print(f"Sample 2 - Label length: {len(batch_samples[1]['labels'])}")

# Apply collator
collated_batch = data_collator(batch_samples)

print("\n=== SETELAH DATACOLLATOR ===")
print(f"Batch input_ids shape: {collated_batch['input_ids'].shape}")
print(f"Batch labels shape: {collated_batch['labels'].shape}")

# Analisis sample pertama
first_labels = collated_batch['labels'][0]
print(f"\nFirst sample labels (30 pertama): {first_labels[:30]}")

# Hitung masked vs non-masked
masked_count = (first_labels == -100).sum().item()
non_masked_count = (first_labels != -100).sum().item()
total_count = len(first_labels)

print(f"\n=== ANALISIS MASKING ===")
print(f"Masked tokens (-100): {masked_count} ({masked_count/total_count*100:.1f}%)")
print(f"Non-masked tokens: {non_masked_count} ({non_masked_count/total_count*100:.1f}%)")
print(f"Total tokens: {total_count}")

if non_masked_count == 0:
    print("\n❌ MASALAH KRITIS: Semua labels di-mask dengan -100!")
    print("   Ini akan menyebabkan zero loss saat training.")
    print("\n   KEMUNGKINAN PENYEBAB:")
    print("   1. Labels sudah berisi padding (0) sebelum collator")
    print("   2. DataCollator salah mengidentifikasi semua token sebagai padding")
elif masked_count > non_masked_count:
    print("\n⚠️ PERINGATAN: Lebih banyak masked tokens daripada non-masked!")
    print("   Ini tidak normal. Periksa panjang sequence.")
else:
    print(f"\n✓ DataCollator bekerja dengan benar!")
    print(f"   {non_masked_count} valid labels untuk training.")

=== SEBELUM DATACOLLATOR ===
Sample 1 - Input length: 319
Sample 1 - Label length: 201
Sample 2 - Input length: 50
Sample 2 - Label length: 30

=== SETELAH DATACOLLATOR ===
Batch input_ids shape: torch.Size([2, 320])
Batch labels shape: torch.Size([2, 208])

First sample labels (30 pertama): tensor([  926,   369,    23,    30,    39, 15506,  1489, 10059,    10,   138,
        19133,   211, 22502,    21, 12942,     8,  2046,    43,  1620,   614,
           21,  6983,  1547,  6273, 19133,    39,  7676,   120,  7809,    32])

=== ANALISIS MASKING ===
Masked tokens (-100): 7 (3.4%)
Non-masked tokens: 201 (96.6%)
Total tokens: 208

✓ DataCollator bekerja dengan benar!
   201 valid labels untuk training.


## Step 5: Test Forward Pass

Verifikasi bahwa model menghasilkan loss yang valid (bukan 0.0 atau NaN).

In [16]:
# Pindahkan batch ke device
device = peft_model.device
batch_on_device = {
    k: v.to(device) if isinstance(v, torch.Tensor) else v 
    for k, v in collated_batch.items()
}

# Forward pass
peft_model.eval()
with torch.no_grad():
    outputs = peft_model(**batch_on_device)

print("=== HASIL FORWARD PASS ===")
print(f"\nLoss: {outputs.loss.item():.4f}")
print(f"Logits shape: {outputs.logits.shape}")

if outputs.loss.item() == 0.0:
    print("\n❌ MASALAH: Loss adalah 0.0!")
    print("   Ini mengkonfirmasi masalah label masking.")
    print("   Model tidak menerima learning signal.")
elif torch.isnan(outputs.loss):
    print("\n❌ MASALAH: Loss adalah NaN!")
    print("   Ini menunjukkan numerical instability.")
    print("   Kemungkinan: learning rate terlalu tinggi atau gradient explosion.")
else:
    print(f"\n✓ Loss valid: {outputs.loss.item():.4f}")
    print("   Model dapat menerima learning signal.")

=== HASIL FORWARD PASS ===

Loss: 9.9250
Logits shape: torch.Size([2, 208, 32128])

✓ Loss valid: 9.9250
   Model dapat menerima learning signal.


## Step 6: Test Generation

Verifikasi bahwa model dapat generate output yang masuk akal.

In [17]:
# Generate prediction
input_ids = collated_batch['input_ids'][0:1].to(device)
attention_mask = collated_batch['attention_mask'][0:1].to(device)

with torch.no_grad():
    generated_ids = peft_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )

# Decode
prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
reference = sample['target']

print("=== TEST GENERATION ===")
print(f"\nInput (200 char): {sample['input'][:200]}...")
print(f"\nReference (200 char): {reference[:200]}...")
print(f"\nPrediction (200 char): {prediction[:200]}...")
print(f"\nPrediction length: {len(prediction)} chars")

# Analisis kualitas prediction
if len(prediction) < 10:
    print("\n❌ MASALAH: Prediction terlalu pendek!")
    print("   Model tidak generate output yang bermakna.")
elif prediction.lower()[:50] in sample['input'].lower():
    print("\n❌ MASALAH: Model meng-copy input!")
    print("   Model tidak belajar task AQG.")
elif not prediction.lower().startswith('pertanyaan'):
    print("\n⚠️ PERINGATAN: Prediction tidak dimulai dengan 'Pertanyaan'")
    print("   Format output mungkin tidak sesuai.")
else:
    print("\n✓ Prediction terlihat reasonable")
    print("   Model generate output dengan format yang benar.")

=== TEST GENERATION ===

Input (200 char): Konteks: ### Perbandingan Penggunaan Memori

```python
import numpy
import sys

var_list = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
var_array = numpy.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("Ukuran k...

Reference (200 char): Pertanyaan: Sesuai catatan modul yang menggunakan list Python untuk matriks, lengkapi kode berikut untuk menghitung ukuran memori list: import sys; var_list = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]; ukuran...

Prediction (200 char): Perbandingan Penggunaan Memori Konteks: ### Perbandingan Penggunaan Memori Konteks: ### Perbandingan Penggunaan Memori Konteks: ### Perbandingan Penggunaan Memori Konteks: ### Perbandingan Penggunaan ...

Prediction length: 414 chars

⚠️ PERINGATAN: Prediction tidak dimulai dengan 'Pertanyaan'
   Format output mungkin tidak sesuai.


## Summary dan Rekomendasi

Berdasarkan hasil debug di atas:

### Diagnosis

1. **Jika semua labels adalah 0**: 
   - Masalah di tokenization
   - Periksa parameter `text_target`

2. **Jika semua labels adalah -100**: 
   - Masalah di DataCollator
   - Periksa `label_pad_token_id`

3. **Jika loss adalah 0.0**: 
   - Konfirmasi masalah label masking
   - Model tidak menerima learning signal

4. **Jika prediction adalah gibberish**: 
   - Model tidak belajar
   - Perlu domain adaptation

### Langkah Selanjutnya

1. **Fix masalah yang teridentifikasi**
2. **Jalankan domain adaptation** (340 samples, 2 epochs)
3. **Augment dataset** ke 2000+ samples
4. **Adjust hyperparameters**:
   - LoRA r: 16 (dari 8)
   - Learning rate: 5e-5 (dari 1e-4)
   - Epochs: 5 (dari 3)

### Referensi

- [DataCollatorForSeq2Seq Docs](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorForSeq2Seq)
- [T5 Tokenizer Docs](https://huggingface.co/docs/transformers/model_doc/t5#transformers.T5Tokenizer)
- [Training Report](../../docs/training-report/indot5-report2.md)